# 04 — Model Evaluation, Residuals & Decadal Forecast
**Global Temperature Forecasting Using Multimodal Machine Learning**  
Choudhary & Kulkarni (2026), *Climatic Change* (Springer)  

Reproduces **§6–§9** and **Figures 8–10**, **Table 4**:
- §6: Nine-model benchmark on test set (2018–2025)
- §7: Training dynamics and convergence analysis
- §8: CNN-LSTM residual diagnostics (Q-Q, heteroscedasticity, histogram)
- §9: 10-year ensemble decadal projections (2025–2035) vs. Paris 1.5°C

**Best model: CNN-LSTM**  
MAE=0.0944°C | RMSE=0.1175°C | R²=0.6781 | Murphy Skill=0.7462

In [ ]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path

from preprocessing import load_all
from feature_engineering import prepare_all
from evaluation import (
    evaluate_single_model, benchmark_table, save_benchmark_table,
    residual_diagnostics, plot_predictions
)
from ensemble_model import load_models, ensemble_forecast, plot_decadal_forecast, save_forecast_csv
from utils import set_global_seed, apply_style, ensure_dirs

set_global_seed(42)
apply_style()
FIG_DIR  = Path('../results/figures'); FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR  = Path('../results/tables');  TAB_DIR.mkdir(parents=True, exist_ok=True)

raw  = load_all(use_cache=True)
data = prepare_all(raw, seq_len=36)
X_test, y_test = data['seq_test']
scaler = data['scaler']
print('Data ready.')

In [ ]:
# ── Cell 2: Load trained models ──────────────────────────────────────────────
trained_models = load_models(
    ['CNN-LSTM', 'GRU', 'LSTM', 'Bi-LSTM', 'Transformer']
)
print(f'Loaded models: {list(trained_models.keys())}')
# NOTE: If models not yet trained, run notebook 03 first.

In [ ]:
# ── Cell 3: Evaluate all DL models on test set ───────────────────────────────
y_train_orig = scaler.inverse_y(scaler.target_scaler.transform(
    data['train_df'][['temp_anomaly']].values).ravel())

dl_results = {}
for name, model in trained_models.items():
    dl_results[name] = evaluate_single_model(
        model, X_test, y_test, scaler, name, y_train_orig
    )
print('\nDL model evaluation complete.')

In [ ]:
# ── Cell 4: Nine-model benchmark table (Table 4) ─────────────────────────────
# Add baseline results (from notebook 03 / train_baselines.py)
import joblib
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from utils import murphy_skill_score

all_results = dict(dl_results)

# Load baselines if available
baseline_names = {'LightGBM':'lightgbm_model.pkl','XGBoost':'xgboost_model.pkl',
                  'Random Forest':'random_forest_model.pkl','Ridge':'ridge_model.pkl'}
X_test_flat = scaler.feat_scaler.transform(data['test_df'][data['feat_cols']].values)
y_true_flat = scaler.inverse_y(
    scaler.target_scaler.transform(data['test_df'][['temp_anomaly']].values).ravel())

for bname, bfile in baseline_names.items():
    bpath = Path('../models') / bfile
    if bpath.exists():
        mdl = joblib.load(bpath)
        pred_s = mdl.predict(X_test_flat)
        pred   = scaler.inverse_y(pred_s)
        all_results[bname] = {
            'MAE': mean_absolute_error(y_true_flat, pred),
            'RMSE': np.sqrt(mean_squared_error(y_true_flat, pred)),
            'MAPE': float('nan'),
            'R2': r2_score(y_true_flat, pred),
            'Skill': murphy_skill_score(y_true_flat, pred),
            'pred': pred, 'true': y_true_flat
        }

tbl = save_benchmark_table(all_results, str(TAB_DIR/'benchmark_results.csv'))
display(tbl)

In [ ]:
# ── Cell 5: Predictions vs observed (Figure 8) ───────────────────────────────
test_dates = data['test_df']['date'].iloc[36:].reset_index(drop=True)
fig = plot_predictions(dl_results, test_dates, save_as=None)
fig.savefig(FIG_DIR/'fig8_test_predictions.png', dpi=300, bbox_inches='tight')
plt.show()
print('Figure 8 saved.')

In [ ]:
# ── Cell 6: Residual diagnostics for CNN-LSTM (Figure 9) ────────────────────
best = dl_results['CNN-LSTM']
fig, summary = residual_diagnostics(
    y_true=best['true'], y_pred=best['pred'],
    model_name='CNN-LSTM', save_as=None
)
fig.savefig(FIG_DIR/'fig9_residual_diagnostics.png', dpi=300, bbox_inches='tight')
plt.show()
print('\nResidual summary (paper §8):')
for k, v in summary.items():
    print(f'  {k}: {v:.4f}')

In [ ]:
# ── Cell 7: 10-year ensemble forecast 2025–2035 (Figure 10) ─────────────────
ensemble_members = {k: v for k, v in trained_models.items() if k != 'Transformer'}
forecasts, ensemble_mean, future_dates = ensemble_forecast(
    ensemble_members, data, horizon_years=10
)
fig = plot_decadal_forecast(forecasts, ensemble_mean, future_dates, data, save_as=None)
fig.savefig(FIG_DIR/'fig10_decadal_forecast.png', dpi=300, bbox_inches='tight')
plt.show()

df_fc = save_forecast_csv(forecasts, ensemble_mean, future_dates)
print('\n10-Year Forecast Summary:')
print(f'  2025 start : {ensemble_mean[0]:.3f}°C')
print(f'  2035 end   : {ensemble_mean[-1]:.3f}°C')
print(f'  Paris 1.5°C exceeded: {(ensemble_mean >= 1.5).any()}')
print('\nFigure 10 saved. Forecast table saved to results/tables/')

In [ ]:
# ── Cell 8: Ablation study — feature group importance ───────────────────────
# Tests performance impact of removing each feature group
# (Results reported in supplementary material)
feature_groups = {
    'All features'        : data['feat_cols'],
    'No CO₂ features'     : [c for c in data['feat_cols'] if 'co2' not in c and 'forcing' not in c],
    'No solar features'   : [c for c in data['feat_cols'] if 'ssn' not in c and 'sunspot' not in c],
    'No lag features'     : [c for c in data['feat_cols'] if 'lag' not in c and 'ma' not in c],
    'No interaction terms': [c for c in data['feat_cols'] if 'interact' not in c and 'x_' not in c],
}
print('Feature groups for ablation study:')
for name, cols in feature_groups.items():
    print(f'  {name:<22}: {len(cols)} features')
print('\n(Run full ablation via ablation_study.py for Table S1)')